In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Pep-Lab_db)

This notebook curates the **Pep-Lab_db** dataset by consolidating toxic peptide sequences provided in both FASTA and Excel formats. \
The two sources are merged into a unified table, standardized, checked for duplicate consistency, and exported together with dataset-level metadata.

- **Toxic effect / endpoint:** toxic
- **Source:** Pep-Lab_db
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads toxic peptide sequences from two formats**:
  - a FASTA file (`PepLab_Toxin.fasta`),
  - an Excel spreadsheet (`PepLab_Toxin.xlsx`).
- **Assigns labels**:
  - all sequences are treated as toxic positives (`label = 1`).
- **Concatenates sources** into a single DataFrame with a standardized schema:
  - `sequence`
  - `label`
- **Checks duplicated sequences**:
  - identical sequences are collapsed when consistent,
  - any conflicting cases are flagged and exported as errors.
- **Builds metadata** using the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`,
  - `metadata.json`.

In [2]:
name_source = "Pep-Lab_db"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_fasta = (
    read_fasta_doc(f"{PATH_INPUT}/{name_source}/PepLab_Toxin.fasta")
    .assign(label=1)
    [["sequence", "label"]]
)

In [4]:
df_xlsx = (
    pd.read_excel(f"{PATH_INPUT}/{name_source}/PepLab_Toxin.xlsx")
    .assign(label=1)
    [["sequence", "label"]]

)

- Concatenating dataset

In [5]:
df_pep_lab_db = pd.concat([df_fasta, df_xlsx], ignore_index=True)
df_pep_lab_db.shape

(502, 2)

- Checking duplicates

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_pep_lab_db, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(249, 2)

In [8]:
df_errors.shape

(0, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_pep_lab_db)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Dynamic',
 'license': 'No information',
 'year of publication': 2023,
 'last update date': datetime.datetime(2023, 1, 10, 0, 0),
 'download date': Timestamp('2025-06-27 00:00:00'),
 'file format': 'fasta;xlsx',
 'peptide property': 'toxic',
 'dataset information': 'Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://www.pep-lab.info/downloads',
 'publication': 'https://www.mdpi.com/2076-3417/13/2/961',
 'number_of_raw_sequences': 502,
 'number_of_sequences_retained': 249,
 'number_of_positive_sequences': 249,
 'number_of_negative_sequences': 0,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)